In [16]:
import json
from pathlib import Path
import pandas as pd

RESULTS_DIR_MAIN = Path('results')
RESULTS_DIR_SANITY = Path('results')
METHODS = ['Entropy', 'MC_Dropout', 'BNN', 'DRUE']

# Main experiments (exp1-8) + Tier-1 sanity checks (exp9-10)
EXPERIMENTS_MAIN = [
    'exp1_3T3_to_3T3_scaffold',
    'exp2_3T3_to_3T3_tanimoto',
    'exp3_3T3_to_HEK_scaffold',
    'exp4_3T3_to_HEK_tanimoto',
    'exp5_HEK_to_HEK_scaffold',
    'exp6_HEK_to_HEK_tanimoto',
    'exp7_HEK_to_3T3_scaffold',
    'exp8_HEK_to_3T3_tanimoto',
]

EXPERIMENTS_SANITY = [
    'exp9_3T3_to_random_tanimoto',
    'exp10_HEK_to_random_tanimoto',
]

def load_experiments(exp_list, results_dir):
    records = []
    for exp in exp_list:
        json_path = results_dir / exp / 'ood_roc.json'
        if not json_path.exists():
            print(f'Missing: {json_path}')
            continue
        data = json.load(open(json_path))
        row = {'experiment': exp}
        for method in METHODS:
            if method in data:
                row[f'{method}_AUC']        = data[method]['auc']
                row[f'{method}_AVG_UE_ID']  = data[method]['avg_uncertainty_id']
                row[f'{method}_AVG_UE_OOD'] = data[method]['avg_uncertainty_ood']
            else:
                row[f'{method}_AUC']        = None
                row[f'{method}_AVG_UE_ID']  = None
                row[f'{method}_AVG_UE_OOD'] = None
        records.append(row)
    return pd.DataFrame(records).set_index('experiment')

df_main   = load_experiments(EXPERIMENTS_MAIN,   RESULTS_DIR_MAIN)
df_sanity = load_experiments(EXPERIMENTS_SANITY, RESULTS_DIR_SANITY)
df_flat   = pd.concat([df_main, df_sanity])
print(f'Loaded {len(df_main)} main + {len(df_sanity)} sanity experiments')

Loaded 8 main + 2 sanity experiments


In [17]:
# ── Table 1: AUC ─────────────────────────────────────────────────────────────
# Rows = experiments, Columns = 4 methods

df_auc = df_flat[[f'{m}_AUC' for m in METHODS]].copy()
df_auc.columns = METHODS

def highlight_best_auc(row):
    numeric = row.dropna()
    if numeric.empty:
        return ['' for _ in row]
    best = numeric.max()
    return ['font-weight: bold; background-color: #d4edda' if v == best else '' for v in row]

print('Table 1: OOD Detection AUC (higher = better, best per row in green)')
df_auc.style \
    .format('{:.4f}', na_rep='—') \
    .apply(highlight_best_auc, axis=1) \
    .set_caption('OOD Detection AUC')

Table 1: OOD Detection AUC (higher = better, best per row in green)


,Entropy,MC_Dropout,BNN,DRUE
experiment,,,,
exp1_3T3_to_3T3_scaffold,0.5178,0.5133,0.5161,0.5248
exp2_3T3_to_3T3_tanimoto,0.6060,0.5857,0.6073,0.6408
exp3_3T3_to_HEK_scaffold,0.5062,0.4985,0.5054,0.5164
exp4_3T3_to_HEK_tanimoto,0.5869,0.5625,0.5840,0.5492
exp5_HEK_to_HEK_scaffold,0.5049,0.5050,0.5046,0.5307
exp6_HEK_to_HEK_tanimoto,0.5695,0.5618,0.5657,0.6423
exp7_HEK_to_3T3_scaffold,0.5120,0.5138,0.5127,0.4845
exp8_HEK_to_3T3_tanimoto,0.5767,0.5753,0.5792,0.6383
exp9_3T3_to_random_tanimoto,0.4784,0.4831,0.4909,0.9516


In [18]:
# ── Table 2: AVG_UE_ID and AVG_UE_OOD ────────────────────────────────────────
# Columns level 0 = 4 methods
# Columns level 1 = AVG_UE_ID / AVG_UE_OOD
# Expected: OOD > ID for each method (model detects shift)

col_tuples = [(m, metric)
              for m in METHODS
              for metric in ['AVG_UE_ID', 'AVG_UE_OOD']]
df_ue = df_flat[[f'{m}_{k}' for m, k in col_tuples]].copy()
df_ue.columns = pd.MultiIndex.from_tuples(col_tuples)

def highlight_ood_gt_id(df):
    """Green if OOD > ID, red if OOD <= ID."""
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for method in METHODS:
        id_col  = (method, 'AVG_UE_ID')
        ood_col = (method, 'AVG_UE_OOD')
        for exp in df.index:
            id_val  = df.loc[exp, id_col]
            ood_val = df.loc[exp, ood_col]
            if pd.isna(id_val) or pd.isna(ood_val):
                continue
            color = 'background-color: #d4edda' if ood_val > id_val else 'background-color: #f8d7da'
            styles.loc[exp, id_col]  = color
            styles.loc[exp, ood_col] = color
    return styles

print('Table 2: Average Uncertainty (green = OOD > ID ✓, red = OOD <= ID ✗)')
df_ue.style \
    .format('{:.6f}', na_rep='—') \
    .apply(highlight_ood_gt_id, axis=None) \
    .set_caption('Average Uncertainty: ID vs OOD')

Table 2: Average Uncertainty (green = OOD > ID ✓, red = OOD <= ID ✗)
